## 1 - Loading Processed Data

In this step, we load processed video dataset from the previously notebooks


In [2]:
import pandas as pd
import os

In [4]:
print("--- LOADING PROCESSED DATA ---")

load_path = "./processed_images/fplusplus_extracted_frames.csv"
fplusplus_imgs = pd.read_csv(load_path)

print(f"{len(fplusplus_imgs)} loaded!")
pd.set_option('display.max_colwidth', None)
display(fplusplus_imgs.sample(5))

load_path = "./processed_images/fei_images_final.csv"
fei_imgs = pd.read_csv(load_path)

print(f"{len(fei_imgs)} images loaded!")
pd.set_option('display.max_colwidth', None)
display(fei_imgs.sample(5))

load_path = "./processed_images/celeb_frames.csv"
celeb_imgs = pd.read_csv(load_path)

print(f"{len(celeb_imgs)} images loaded!")
pd.set_option('display.max_colwidth', None)
display(celeb_imgs.sample(5))

--- LOADING PROCESSED DATA ---
70000 loaded!


,path,label,split,dataset,method
42567,F++_Split_Intra/source_based/train/original/original_474_f2_ssd.jpg,0,train,FF++,source_based
60080,F++_Split_Intra/source_based/train/fake/Deepfakes_269_268_f1_ssd.jpg,1,train,FF++,source_based
30935,F++_Split_Intra/target_based/val/fake/Deepfakes_936_931_f4_ssd.jpg,1,val,FF++,target_based
13411,F++_Split_Intra/target_based/train/fake/FaceShifter_991_064_f4_ssd.jpg,1,train,FF++,target_based
9581,F++_Split_Intra/target_based/train/fake/DeepFakeDetection_03_01__outside_talking_still_laughing__JZUXXFRB_f3_ssd.jpg,1,train,FF++,target_based


8054 images loaded!


,path,label,split,dataset,method
4263,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI_Split_Intra/train/fake/M_110-11_194-11_C05_B50_W50_PA05_PM00_F00_ssd.png,1,train,FEI,NaN
3143,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI_Split_Intra/train/fake/M_78-11_69-11_C03_B50_W50_PA03_PM00_F00_ssd.png,1,train,FEI,NaN
3696,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI_Split_Intra/train/fake/M_62-11_171-11_C15_B50_W50_PA15_PM00_F00_ssd.png,1,train,FEI,NaN
7375,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI_Split_Intra/train/fake/M_22-11_5-11_C02_B30_W30_PA02_PM00_F00_ssd.png,1,train,FEI,NaN
1567,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/FEI_Split_Intra/train/fake/M_55-11_49-11_C02_B30_W30_PA02_PM00_F00_ssd.png,1,train,FEI,NaN


1780 images loaded!


,path,label,split,dataset,method
600,CelebDF_Test/original/id19_id19_original_id19_0005_f0_ssd.jpg,0,test,CelebDF,NaN
1771,CelebDF_Test/fake/id23_id6_HifiFace_id6_id23_0001_f0_ssd.jpg,1,test,CelebDF,NaN
844,CelebDF_Test/fake/id26_id9_GHOST_id9_id26_0009_f0_ssd.jpg,1,test,CelebDF,NaN
16,CelebDF_Test/fake/id00812_id21_SadTalker_id21_0003_test_id00812_7ppP2WTpZ1s_f0_ssd.jpg,1,test,CelebDF,NaN
348,CelebDF_Test/fake/id35_id32_LIA_id32_id35_0001_f0_ssd.jpg,1,test,CelebDF,NaN


## 2 - Master Dataset Compilation & Data Export

In this final preprocessing step, we consolidate our distinct datasets into standardized structures:

1. **Format Standardization:** We unify the column structure (`path`, `label`, `split`, `dataset`, `method`) across all dataframes and explicitly cast the labels into standard integers (`0` for Real, `1` for Fake).
2. **Master Dataset Assembly (FEI + FF++):** We concatenate the FEI and FaceForensics++ dataframes into a single, shuffled dataset. This `master_df` is exported as `master_dataset.csv` and contains our primary **Train**, **Validation**, and **Internal Test** splits.
3. **External Test Isolation (Celeb-DF):** The Celeb-DF dataset is deliberately excluded from the master dataset. It is held out completely as an **External Test Set**, serving exclusively for evaluating the model's cross-dataset generalization capabilities.
4. **Final Audit:** We output grouped distribution tables and random samples to verify the final dataset composition, split proportions, and label balancing.

In [ ]:
df_fei = fei_imgs[['path', 'label', 'split']].copy()
df_fei['dataset'] = 'FEI'
df_fei['method'] = 'N/A' 
df_fei['label'] = df_fei['label'].replace({'original': 0, 'fake': 1}).astype(int) 

df_ff = fplusplus_imgs[['path', 'label', 'split', 'method']].copy()
df_ff['dataset'] = 'FF++'
df_ff['label'] = df_ff['label'].astype(int)

df_celeb = celeb_imgs[['path', 'label', 'split']].copy()
df_celeb['label'] = df_celeb['label'].astype(int)

master_df = pd.concat([df_fei, df_ff], ignore_index=True)
master_df = master_df.sample(frac=1, random_state=42).reset_index(drop=True)
master_df.to_csv("master_dataset.csv", index=False)
print("Merge completed! File saved as 'master_dataset.csv'.")

print("--- MASTER DATASET DISTRIBUTION (FEI + FF++) ---")
distribution_master = master_df.groupby(['dataset', 'method', 'split', 'label']).size().unstack(fill_value=0)
if len(distribution_master.columns) == 2:
    distribution_master.columns = ['0 (Real)', '1 (Fake)']
display(distribution_master)

print("\n--- CELEB-DF DISTRIBUTION (EXTERNAL TEST SET) ---")
distribution_celeb = df_celeb.groupby(['path', 'split', 'label']).size().unstack(fill_value=0)
if len(distribution_celeb.columns) == 2:
    distribution_celeb.columns = ['0 (Real)', '1 (Fake)']
display(distribution_celeb)


print("\n--- SAMPLES ---")
print("Master Dataset Sample (Train/Val/Test interni):")
display(master_df.sample(5))

print("\nCeleb-DF Sample (Test esterno):")
display(df_celeb.sample(5))

## 3 - Saving Data & Exporting Archive

To conclude this notebook, we first save our fully cleaned and processed DataFrames to local CSV files (e.g., `master_dataset.csv`, `celeb_test_dataset.csv`). This ensures our prepared metadata is safely stored and ready to be directly loaded into the next notebook of our pipeline without needing to re-run the intensive preprocessing and extraction steps.

Finally, we package the entire structured dataset into a single, highly portable archive (`deepfake_dataset.zip`). This makes it easy to download, store, or transfer the data for model training.

**Contents of the final archive:**
* `F++_Split_Intra/`: The processed and split FaceForensics++ frames.
* `FEI_Split_Intra/`: The processed and split FEI dataset frames.
* `CelebDF_Test/`: The completely held-out Celeb-DF frames for cross-dataset testing.
* `master_dataset.csv`: The unified metadata for the training, validation, and internal test sets.
* `celeb_test_dataset.csv`: The metadata specifically for the external Celeb-DF test set.

*(Note: We use the `-r` flag to include all subdirectories recursively and the `-q` flag to run the compression quietly, keeping the notebook output clean).*

In [ ]:
print("--- SAVING DATAFRAME ---")

save_path_master = "master_dataset.csv"
save_path_test = "celeb_test_dataset.csv"

master_df.to_csv(save_path_master, index=False)
df_celeb.to_csv(save_path_test, index=False)

print(f"Master Data successfully saved to: {save_path_master}")
print(f"Celeb Test Data successfully saved to: {save_path_test}")

In [ ]:
!zip -rq deepfake_dataset.zip F++_Split_Intra FEI_Split_Intra CelebDF_Test master_dataset.csv celeb_test_dataset.csv